# Starter analysis

The basics: loading `results/timings.csv`, turning batch totals into per-operation costs,
summarizing replicates, and estimating a growth rate from a log-log fit.
It uses the rows from the worked `example` experiment (`make experiments ... ONLY=example`).

Make your own notebook(s) in `analysis/` for the real analysis; leave this one as a reference.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RESULTS = Path("..") / "results" / "timings.csv"
df = pd.read_csv(RESULTS)

# Per-operation cost: every row is a batch of n_ops operations.
df["us_per_op"] = df["total_ns"] / df["n_ops"] / 1_000
for counter in ["comparisons", "nodes_visited", "rotations", "splits", "resizes"]:
    df[f"{counter}_per_op"] = df[counter] / df["n_ops"]

df[["experiment", "code_version", "machine_id", "index_type", "operation", "n_keys", "rep", "us_per_op", "comparisons_per_op"]].head()

## Always know which code you're looking at

Rows from before and after a bug fix live in the same file. Filter on `code_version`
(and `machine_id`) before you compare anything.

In [ ]:
df.groupby(["code_version", "machine_id", "experiment"]).size()

In [ ]:
example = df[df["experiment"] == "example"]
if example.empty:
    raise RuntimeError("No 'example' rows yet: run `make experiments MEMBER=you MACHINE=id ONLY=example` first.")
# Pick one code version and one machine to keep the comparison controlled.
version = example["code_version"].iloc[-1]
machine = example["machine_id"].iloc[-1]
example = example[(example["code_version"] == version) & (example["machine_id"] == machine)]

# Summarize replicates with the median and the interquartile range.
summary = (
    example.groupby(["operation", "index_type", "n_keys"])["us_per_op"]
    .describe(percentiles=[0.25, 0.5, 0.75])[["count", "25%", "50%", "75%"]]
    .rename(columns={"50%": "median"})
)
summary

## Cost vs. n on log-log axes

On log-log axes, cost proportional to $n^k$ is a straight line with slope $k$:
roughly 0 for constant time, 1 for linear time, and slowly bending for $\log n$.

In [ ]:
hits = example[example["operation"] == "point_hit"]
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.lineplot(data=hits, x="n_keys", y="us_per_op", hue="index_type",
             estimator="median", errorbar=("pi", 50), marker="o", ax=ax)
ax.set(xscale="log", yscale="log", xlabel="keys in index (n)",
       ylabel="µs per lookup (median, IQR band)", title=f"Point lookups (hits), code {version}")
ax.legend(title="index", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

## Estimating the growth exponent

Fit $\log_{10}(\text{cost}) = a + k \log_{10}(n)$ to the per-n medians. With only three sizes
this is a rough estimate; your experiments should use more sizes, and you should report an
interval for $k$ (e.g. bootstrap over replicates), not just a point estimate.

In [ ]:
rows = []
for index_type, g in hits.groupby("index_type"):
    med = g.groupby("n_keys")["us_per_op"].median()
    k, a = np.polyfit(np.log10(med.index), np.log10(med.values), 1)
    rows.append({"index_type": index_type, "slope_k": round(k, 2)})
pd.DataFrame(rows).sort_values("slope_k")

## Counters vs. timings

Counters (`comparisons_per_op`, `nodes_visited_per_op`, ...) are deterministic, so they have no
replicate-to-replicate noise and are identical on every machine. Plot them against theory,
e.g. $\log_2 n$ for a balanced binary tree. The reference indexes (`ref_*`) don't count operations.

In [ ]:
counted = hits[hits["comparisons"].notna()]
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.lineplot(data=counted, x="n_keys", y="comparisons_per_op", hue="index_type", marker="o", ax=ax)
n = np.array(sorted(counted["n_keys"].unique()))
ax.plot(n, np.log2(n), "k--", label="log2(n)")
ax.set(xscale="log", yscale="log", xlabel="keys in index (n)", ylabel="comparisons per lookup")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()